# Práctica 2: Filtro de Bayes Discreto (Grid Localization)

En esta práctica implementaremos un **Filtro de Bayes Discreto** completo. Este algoritmo es una especialización no paramétrica del filtro de Bayes, donde el espacio de estados se divide en celdas o rejillas discretas.

### Escenario:
Imagina un robot móvil que se desplaza a lo largo de un pasillo circular dividido en 20 celdas discretas. El pasillo contiene tres puertas ubicadas en posiciones fijas conocidas. El robot no sabe dónde se encuentra inicialmente (incertidumbre global uniforme) y debe auto-localizarse combinando un sensor de puertas ruidoso y un actuador impreciso.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# Configuración del gráfico
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

---
## 1. Definición del Mundo Discreto

Nuestro pasillo circular consta de $N=20$ celdas. Vamos a definir las puertas como `1` y las paredes como `0`.
Las puertas estarán situadas en las celdas **3, 9 y 15**.

In [ ]:
N = 20
mapa = np.zeros(N, dtype=int)
mapa[3] = 1   # Puerta 1
mapa[9] = 1   # Puerta 2
mapa[15] = 1  # Puerta 3

def graficar_mundo(belief=None, titulo="Mapa del Pasillo Circular"):
    fig, ax = plt.subplots(figsize=(12, 4.5))
    
    # Dibujar celdas (Paredes vs Puertas)
    for i in range(N):
        color = '#ffbb78' if mapa[i] == 1 else '#d9d9d9'
        label_celda = '🚪' if mapa[i] == 1 else '🧱'
        ax.text(i, 0.2, label_celda, ha='center', va='center', fontsize=18)
        ax.axvline(i - 0.5, color='gray', linestyle=':', alpha=0.5)
        
    if belief is not None:
        # Dibujar barras de probabilidad de creencia
        ax.bar(np.arange(N), belief, color='#1f77b4', alpha=0.75, width=0.8, label='Creencia $bel(x_t)$')
        ax.set_ylabel("Probabilidad")
        ax.legend(loc='upper right', frameon=True, facecolor='white')
        
    ax.set_xticks(np.arange(N))
    ax.set_xlabel("Celdas del Pasillo ($x_i$)")
    ax.set_ylim(-0.1, 1.05)
    ax.set_xlim(-0.7, N - 0.3)
    ax.set_title(titulo, pad=15)
    plt.tight_layout()
    plt.show()

# Inicializamos la creencia con total incertidumbre (Prior Uniforme)
belief_inicial = np.ones(N) / N
graficar_mundo(belief_inicial, titulo="Creencia Inicial: Incertidumbre Global Uniforme")

---
## 2. Paso de Predicción ( Chapman-Kolmogorov Discreto)

Cuando el robot intenta moverse hacia adelante cierta cantidad de celdas ($u_t$), hay imprecisión física. Definiremos las siguientes probabilidades de transición:
- El robot se mueve **exactamente las celdas deseadas** con probabilidad de $0.75$.
- Se queda **corto por 1 celda** con probabilidad de $0.15$.
- Se pasa de largo por **1 celda** con probabilidad de $0.10$.

Dado que el pasillo es **circular**, utilizamos aritmética modular (operador `% N`) para controlar que las partículas y probabilidades vuelvan al inicio.

In [ ]:
def predict(belief, u):
    """
    Paso de Predicción del Filtro de Bayes Discreto.
    Desplaza la creencia 'u' posiciones a la derecha aplicando el ruido del movimiento.
    """
    N = len(belief)
    belief_pred = np.zeros(N)
    
    for i in range(N):
        # El robot pudo haber llegado a la celda 'i' desde tres orígenes posibles:
        
        # 1. Movimiento exacto: origen en (i - u) % N
        origen_exacto = (i - u) % N
        prob_exacta = 0.75
        
        # 2. Se quedó corto: origen en (i - u + 1) % N
        origen_corto = (i - u + 1) % N
        prob_corta = 0.15
        
        # 3. Se pasó de largo: origen en (i - u - 1) % N
        origen_largo = (i - u - 1) % N
        prob_larga = 0.10
        
        # Ecuación de Chapman-Kolmogorov discreta
        belief_pred[i] = (belief[origen_exacto] * prob_exacta +
                          belief[origen_corto] * prob_corta +
                          belief[origen_largo] * prob_larga)
        
    return belief_pred

# Probemos movernos u = 5 a partir de una creencia concentrada al 100% en la celda 5
creencia_test = np.zeros(N)
creencia_test[5] = 1.0

creencia_predicha = predict(creencia_test, u=5)
graficar_mundo(creencia_predicha, titulo="Predicción de Movimiento: u=5 a partir de Celda 5 (Nota la dispersión)")

---
## 3. Paso de Actualización (Teorema de Bayes Discreto)

El robot usa un sensor de proximidad que lee dos posibles medidas: `z = 1` (detecta Puerta) o `z = 0` (detecta Pared).
El sensor es ruidoso, con las siguientes características:
- Si estamos en una **puerta real**, lee "Puerta" con probabilidad $0.80$ (Verdadero Positivo).
- Si estamos en una **pared real**, lee "Puerta" con probabilidad $0.15$ (Falso Positivo).

In [ ]:
def update(belief_bar, z):
    """
    Paso de Actualización (Corrección) del Filtro de Bayes.
    Aplica el modelo del sensor sobre la creencia predicha bel_bar.
    """
    N = len(belief_bar)
    belief_post = np.zeros(N)
    
    # Definir probabilidades del sensor
    p_z_puerta_dado_puerta = 0.80
    p_z_puerta_dado_pared = 0.15
    
    for i in range(N):
        es_puerta = mapa[i] == 1
        
        # Calcular verosimilitud p(z_t | x_t = i)
        if z == 1: # El sensor detectó Puerta
            likelihood = p_z_puerta_dado_puerta if es_puerta else p_z_puerta_dado_pared
        else: # El sensor detectó Pared
            likelihood = (1 - p_z_puerta_dado_puerta) if es_puerta else (1 - p_z_puerta_dado_pared)
            
        # Bayes: Posterior no normalizado = Prior (bel_bar) * Likelihood
        belief_post[i] = belief_bar[i] * likelihood
        
    # Paso de Normalización
    normalizador = np.sum(belief_post)
    if normalizador > 0:
        belief_post /= normalizador
        
    return belief_post

# Probemos medir z = 1 (Puerta) a partir de una creencia inicial uniforme
creencia_post = update(belief_inicial, z=1)
graficar_mundo(creencia_post, titulo="Actualización: El sensor lee 'Puerta' (z=1) con creencia inicial uniforme")

---
## 4. Simulación Completa (El Bucle de Localización)

Vamos a simular un robot real que inicia físicamente en la celda **0** y avanza de celda en celda. El robot no conoce esta posición real inicial.

En cada paso temporizado:
1. El robot avanza de forma ruidosa (pero el simulador calcula su posición real precisa).
2. El sensor toma una lectura ruidosa basada en la posición real del robot.
3. El Filtro de Bayes realiza la Predicción y luego la Corrección.
4. Graficamos la creencia para ver cómo colapsa la incertidumbre global hasta localizar al robot.

In [ ]:
# Inicializaciones
robot_pos_real = 0
bel = np.ones(N) / N # Total incertidumbre al iniciar

# Simulación de 6 pasos de tiempo
pasos = [
    (1, "El robot avanza 1 paso y mide pared"),
    (2, "El robot avanza 2 pasos (celda real 3: ¡Puerta!) y mide puerta"),
    (3, "El robot avanza 3 pasos (celda real 6) y mide pared"),
    (3, "El robot avanza 3 pasos (celda real 9: ¡Puerta!) y mide puerta"),
    (3, "El robot avanza 3 pasos (celda real 12) y mide pared"),
    (3, "El robot avanza 3 pasos (celda real 15: ¡Puerta!) y mide puerta")
]

print("🟢 INICIANDO BUCLE DE LOCALIZACIÓN RECURSIVO...")
graficar_mundo(bel, titulo="Paso 0: Creencia Inicial")

for t_step, (u, descripcion) in enumerate(pasos, 1):
    # 1. Simular movimiento real del robot en el simulador
    robot_pos_real = (robot_pos_real + u) % N
    
    # 2. Simular lectura real del sensor (ruidosa, pero congruente)
    es_puerta_real = mapa[robot_pos_real] == 1
    # Generar lectura
    if es_puerta_real:
        z = 1 if np.random.rand() < 0.80 else 0
    else:
        z = 1 if np.random.rand() < 0.15 else 0
        
    # 3. FILTRO DE BAYES
    bel_bar = predict(bel, u)
    bel = update(bel_bar, z)
    
    # Graficar
    label_titulo = f"Paso {t_step}: {descripcion} | z={z} (Posición Real: {robot_pos_real})"
    graficar_mundo(bel, titulo=label_titulo)
    
    # Mostramos estadísticas de la estimación
    pos_estimada = np.argmax(bel)
    prob_estimada = bel[pos_estimada] * 100
    print(f"⏱️ Paso {t_step} -> Posición Real: {robot_pos_real} | Estimación Máxima: Celda {pos_estimada} con {prob_estimada:.1f}% de confianza.")
    time.sleep(0.5)

---
## 🧪 Reto Práctico para el Alumno

1. **Cambio de Sensores:** Modifica el modelo de sensor en la función `update`. ¿Qué ocurre si el sensor se vuelve extremadamente ruidoso (p. ej., Falso Positivo de $0.45$ y Verdadero Positivo de $0.55$)? ¿Puede el robot localizarse?
2. **Pérdida de Información:** Modifica la probabilidad de movimiento en `predict` para que sea altamente imprecisa ($50\%$ de quedarse corto y $50\%$ de pasarse de largo). Observa cómo se propaga la creencia y la rapidez con la que se 'aplana' la distribución si no hay lecturas de sensor frecuentes.